# L1ME-AID Filter Evaluation Notebook

Evaluate individual filters on L1ME-AID polymorphic TE workflow.

**Filters evaluated:**
1. Individual flags (10 unique flags)
2. Tail Type (`No_Tail_Type` filter)
3. Logsdon filter (element-type-specific coverage/divergence thresholds)
4. Modified Logsdon filter (flank-corrected element percentage)
5. Copy filtration (remove elements already present in reference)

## 1. Load Data

The data only includes a set of non-reference structural variants, along with flanking sequence of 1000bp of reference sequence at both ends, analyzed by L1ME-AID. I evaluate the different filters of L1ME-AID to see why I'm not calling enough TEs.

Before this step, I've included a set of reference sequences to go along with it. Think of it like:

ALT record: ---ref--- + non-ref INS + ---ref---

NULL record: ---ref--- + no sequence included + ---ref---

The goal was to ensure that the reference sequence caught any reference TEs nearby. Imagine that a structural variant caller finds a variant but places the start/end position of the variant well into reference sequence. It is possible that, because of repetitive regions, that the variant caller thinks the reference sequence is part of the variant, and that any TE-like sequences in the reference are actually a de novo TE variant.

The goal was thus to use the NULL record--with only the flanking reference regions--as a null that we could "subtract" out from the ALT record. That is, any TE sequences found in both the ALT and NULL records should be removed, such that only genuinely novel TEs found in the ALT record remain.

Matching variants like this neatly turned out to be a difficult challenge. I ended up making a bash script that works directly on the output of RepeatMasker, instead of L1ME-AID, which tries to bundle things up such that only one repeat is annotated per sequence. I use RepeatMasker to find repeats that have similar coordinates, similar designations, and matching sequences (that is, every ALT record has a corresponding NULL record, so only compare between them).

This filtration notebook describes the remaining filters on all of the unmatched variants. Having a better matching algorithm improved the yield of variants. However, many of these filters still cause a low yield of TEs when L1ME-AID is run on this dataset. This notebook describes how I tried to explore the issue.

This documentation was written retrospectively, so for anyone reading you should know that matching filtration ended up not being that important, and it is simply better to directly feed PanGenie's variant calls straight to RepeatMasker+L1ME-AID instead of adding flank (the issue of variant boundaries in repeats is not an issue with a good genotyper like PanGenie). In fact, adding the repeats causes issues in the length of the sequence fed to L1ME-AID, causing it to throw up many problematic flags described below. Simply giving the ALT variant directly saves you some trouble. Fortunately, polymorphic TEs should insert largely intact, and giving the ALT variant directly should yield most of the relvant TEs that L1ME-AID tries to identify.

In [3]:
import pandas as pd
import ast
import os

my_headers = ["ID", "Sequence", "Element Hits", "Sequence Length", "Element Annotation",
              "Element Divergence", "Element Proportion", "Element Percentage", "Element Designation",
              "Orientation", "Flags", "Tail Beginning", "Tail Type", "Tail Length", "Tail Seed Hits",
              "Unique Element Count", "Twin Priming Flag"]

F1 = pd.read_table("F1_alt_filtered.outFile",
              sep="\t",
              skiprows=[0],
              usecols=range(len(my_headers)),
              header=None,
              names=my_headers)

print(f"F1 shape: {F1.shape}")

F1 shape: (1610, 17)


## 2. Configuration

In [4]:
# Random subsample size per filter per direction (pass/fail)
SUBSAMPLE_N = 50

# Random seed for reproducibility
RANDOM_SEED = 42

# Output directory for TSVs
OUTPUT_DIR = "filter_eval_tsvs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Columns to keep in output (drop Sequence to keep files small)
OUTPUT_COLS = ["ID", "Element Hits", "Sequence Length", "Element Annotation",
               "Element Divergence", "Element Proportion", "Element Percentage",
               "Element Designation", "Orientation", "Flags", "Tail Beginning",
               "Tail Type", "Tail Length", "Tail Seed Hits",
               "Unique Element Count", "Twin Priming Flag"]

## 3. Helper Functions

In [5]:
def subsample_and_write(df_pass, df_fail, filter_name, output_dir=OUTPUT_DIR,
                        n=SUBSAMPLE_N, seed=RANDOM_SEED, cols=OUTPUT_COLS):
    """
    Subsample pass/fail sets and write to TSV files.
    Returns a summary dict.
    """
    # Subsample (or take all if fewer than n)
    n_pass = min(n, len(df_pass))
    n_fail = min(n, len(df_fail))
    
    pass_sample = df_pass.sample(n=n_pass, random_state=seed) if n_pass > 0 else df_pass
    fail_sample = df_fail.sample(n=n_fail, random_state=seed) if n_fail > 0 else df_fail
    
    # Sanitize filter name for filename
    safe_name = filter_name.replace('/', '_').replace('>', 'gt').replace(' ', '_')
    
    pass_path = os.path.join(output_dir, f"{safe_name}_PASS_FILTERED.tsv")
    fail_path = os.path.join(output_dir, f"{safe_name}_FAIL_FILTERED.tsv")
    
    pass_sample[cols].to_csv(pass_path, sep="\t", index=False)
    fail_sample[cols].to_csv(fail_path, sep="\t", index=False)
    
    summary = {
        "filter": filter_name,
        "total_pass": len(df_pass),
        "total_fail": len(df_fail),
        "sampled_pass": n_pass,
        "sampled_fail": n_fail,
        "pass_file": pass_path,
        "fail_file": fail_path
    }
    
    print(f"  {filter_name}: {len(df_pass)} pass, {len(df_fail)} fail "
          f"(sampled {n_pass}/{n_fail})")
    
    return summary


def parse_flags(flag_str):
    """
    Parse a Flags field into a set of individual flags.
    Handles 'No_Flags' and bracketed list strings.
    """
    if flag_str == 'No_Flags':
        return set()
    try:
        return set(ast.literal_eval(flag_str))
    except:
        return {flag_str}

## 4. Work with ALT rows only

In previous iterations of filtration, I remove any rows that include SVs that are in the reference set to be evaluated as filters for non-reference inserts. That means the filters are evaluated on ALT rows (the variant-containing sequences).

Here, of course, there are no NULL rows, as I had already done the matching analysis.

In [6]:
# Separate ALT and NULL rows from the main file
F1_alt = F1

print(f"ALT rows to evaluate: {len(F1_alt)}")

# Parse flags for ALT rows
F1_alt['parsed_flags'] = F1_alt['Flags'].apply(parse_flags)

ALT rows to evaluate: 1610


## 5. Filter Evaluation: Individual Flags

For each unique flag, split ALT rows into:
- **PASS**: rows that do NOT have this flag
- **FAIL**: rows that DO have this flag

L1ME-AID recommends that you filter all rows that have any flags at all or have a No_Tail_Type designation in their Tail_Type feature.

In [7]:
# Get all individual flags
all_flags = set()
for flags in F1_alt['parsed_flags']:
    all_flags.update(flags)

print(f"Individual flags to evaluate: {sorted(all_flags)}\n")

flag_summaries = []

for flag in sorted(all_flags):
    has_flag = F1_alt['parsed_flags'].apply(lambda x: flag in x)
    
    df_pass = F1_alt[~has_flag]  # Does NOT have this flag -> passes filter
    df_fail = F1_alt[has_flag]   # Has this flag -> filtered out
    
    summary = subsample_and_write(df_pass, df_fail, f"flag_{flag}")
    flag_summaries.append(summary)

Individual flags to evaluate: ['ALU_>500Len', 'ALU_>Len', 'ALU_DIVERGENCE', 'Alu_Linker_Region_Warning', 'Bad_Tail_Position', 'LINE_DIVERGENCE', 'MULTI-ALU_Hits', 'No_Tail_Detected', 'OLDER_LINE_SUBFAMILY']

  flag_ALU_>500Len: 1140 pass, 470 fail (sampled 50/50)
  flag_ALU_>Len: 1456 pass, 154 fail (sampled 50/50)
  flag_ALU_DIVERGENCE: 1420 pass, 190 fail (sampled 50/50)
  flag_Bad_Tail_Position: 627 pass, 983 fail (sampled 50/50)
  flag_LINE_DIVERGENCE: 1559 pass, 51 fail (sampled 50/50)
  flag_MULTI-ALU_Hits: 1565 pass, 45 fail (sampled 50/45)
  flag_No_Tail_Detected: 1515 pass, 95 fail (sampled 50/50)
  flag_OLDER_LINE_SUBFAMILY: 1499 pass, 111 fail (sampled 50/50)


In [8]:
print("Tail Type value counts:")
print(F1_alt['Tail Type'].value_counts())
print()

no_tail_mask = F1_alt['Tail Type'] == 'No_Tail_Type'

df_pass = F1_alt[~no_tail_mask]  # Has a tail type -> passes
df_fail = F1_alt[no_tail_mask]   # No tail type -> filtered out

tail_summary = subsample_and_write(df_pass, df_fail, "tail_type_NoTailType")

Tail Type value counts:
Tail Type
Possible_A-Tail*_and_Possible_T-Tail         588
Possible_A-Tail_and_Possible_T-Tail*         571
No_Tail_Type                                 254
Possible_A-Tail                              101
Possible_T-Tail                               89
Possible_MutatedOrTruncated_A-Tail_T-Tail      5
Possible_MutatedOrTruncated_A-Tail             2
Name: count, dtype: int64

  tail_type_NoTailType: 1356 pass, 254 fail (sampled 50/50)


## 7. Filter Evaluation: Logsdon Filter

This filter follows recommendations in the Supplementary information of Logsdon (2025), the paper out of the HGSVC that defined the L1ME-AID workflow and provided much of the structural variation information for the Human Pangenome. These are their criteria for a polymorphic TE:

Element-type-specific thresholds:
- **Alu**: ≥70% coverage AND <6% divergence
- **L1/SVA**: ≥20% coverage AND <15% divergence
- Must be Alu, L1, or SVA (others fail)

In [9]:
is_alu = F1_alt['Element Designation'].str.contains('Alu', case=False, na=False)
is_l1 = F1_alt['Element Designation'].str.contains('L1', case=False, na=False)
is_sva = F1_alt['Element Designation'].str.contains('SVA', case=False, na=False)
is_line_sva = is_l1 | is_sva

# Alu criteria
alu_coverage = F1_alt['Element Percentage'] >= 0.70
alu_divergence = F1_alt['Element Divergence'] < 6
mask_alu = is_alu & alu_coverage & alu_divergence

# L1/SVA criteria
line_sva_coverage = F1_alt['Element Percentage'] >= 0.20
line_sva_divergence = F1_alt['Element Divergence'] < 15
mask_line_sva = is_line_sva & line_sva_coverage & line_sva_divergence

# Combined
logsdon_pass = (is_line_sva | is_alu) & (mask_alu | mask_line_sva)

df_pass = F1_alt[logsdon_pass]
df_fail = F1_alt[~logsdon_pass]

print(f"Logsdon breakdown:")
print(f"  Alu pass: {mask_alu.sum()}")
print(f"  L1/SVA pass: {mask_line_sva.sum()}")
print(f"  Not Alu/L1/SVA (auto-fail): {(~(is_alu | is_line_sva)).sum()}")
print(f"  Alu/L1/SVA but fail thresholds: {((is_alu | is_line_sva) & ~logsdon_pass).sum()}")
print()

logsdon_summary = subsample_and_write(df_pass, df_fail, "logsdon_filter")

Logsdon breakdown:
  Alu pass: 0
  L1/SVA pass: 162
  Not Alu/L1/SVA (auto-fail): 745
  Alu/L1/SVA but fail thresholds: 703

  logsdon_filter: 162 pass, 1448 fail (sampled 50/50)


## 7b. Filter Evaluation: Modified Logsdon Filter (Flank-Corrected)

The original Logsdon filter uses `Element Percentage`, which was computed against the full sequence length including 2×1000 bp of flanking sequence added before RepeatMasker. This dilutes the percentage for short insertions.

**Correction:** Recalculate element percentage as `(Element Percentage × Sequence Length) / (Sequence Length − 2000)`, capped at 1.0. Then apply the same Logsdon thresholds to the corrected values.

In [10]:
FLANK_TOTAL = 2000  # 1000 bp added to each side

# Parse Element Proportion dict to get the proportion for the annotated element only
def get_annotation_proportion(row):
    """
    Extract the proportion for just the Element Annotation from the
    Element Proportion dict, rather than using Element Percentage
    (which sums ALL elements in the sequence).
    """
    annotation = row['Element Annotation']
    prop_str = row['Element Proportion']
    if prop_str == 'NONE' or annotation == 'No_Element_Annotation':
        return 0.0
    try:
        prop_dict = ast.literal_eval(prop_str)
        return prop_dict.get(annotation, 0.0)
    except:
        return 0.0

# Get the per-element proportion (NOT the total Element Percentage)
annotation_proportion = F1_alt.apply(get_annotation_proportion, axis=1)

# Correct for flanking sequence: proportion was computed against full sequence
# length including 2000bp flank. Rescale to insertion length only.
insertion_length = F1_alt['Sequence Length'] - FLANK_TOTAL
element_bases = annotation_proportion * F1_alt['Sequence Length']
corrected_pct = (element_bases / insertion_length).clip(upper=1.0)

# Show the impact of both corrections
print("Original Element Percentage vs annotation-only proportion vs flank-corrected:")
comparison = pd.DataFrame({
    'Element_Percentage': F1_alt['Element Percentage'],
    'annotation_proportion': annotation_proportion,
    'corrected_pct': corrected_pct,
    'seq_len': F1_alt['Sequence Length'],
    'insertion_len': insertion_length,
    'annotation': F1_alt['Element Annotation'],
})
has_element = F1_alt['Element Annotation'] != 'No_Element_Annotation'
print(comparison[has_element][['Element_Percentage', 'annotation_proportion', 'corrected_pct']].describe().round(4))
print()

# Apply modified Logsdon thresholds with corrected, annotation-only percentage
# (reuses is_alu, is_l1, is_sva, is_line_sva, divergence masks from original Logsdon cell)

# Alu: >=70% corrected coverage AND <6% divergence
mod_alu_coverage = corrected_pct >= 0.70
mod_mask_alu = is_alu & mod_alu_coverage & alu_divergence

# L1/SVA: >=20% corrected coverage AND <15% divergence
mod_line_sva_coverage = corrected_pct >= 0.20
mod_mask_line_sva = is_line_sva & mod_line_sva_coverage & line_sva_divergence

# Combined
mod_logsdon_pass = (is_line_sva | is_alu) & (mod_mask_alu | mod_mask_line_sva)

df_pass_mod = F1_alt[mod_logsdon_pass]
df_fail_mod = F1_alt[~mod_logsdon_pass]

print(f"Modified Logsdon breakdown (annotation-only + flank-corrected):")
print(f"  Alu pass: {mod_mask_alu.sum()} (was {mask_alu.sum()} original)")
print(f"  L1/SVA pass: {mod_mask_line_sva.sum()} (was {mask_line_sva.sum()} original)")
print(f"  Not Alu/L1/SVA (auto-fail): {(~(is_alu | is_line_sva)).sum()}")
print(f"  Total pass: {mod_logsdon_pass.sum()} (was {logsdon_pass.sum()} original)")
print(f"  Difference from original: {mod_logsdon_pass.sum() - logsdon_pass.sum()}")
print()

mod_logsdon_summary = subsample_and_write(df_pass_mod, df_fail_mod, "modified_logsdon_filter")

Original Element Percentage vs annotation-only proportion vs flank-corrected:
       Element_Percentage  annotation_proportion  corrected_pct
count           1101.0000              1101.0000      1101.0000
mean               0.2764                 0.2447         0.8055
std                0.2511                 0.2314         0.2944
min                0.0050                 0.0045         0.0060
25%                0.1304                 0.1275         0.7231
50%                0.1368                 0.1341         0.9583
75%                0.3720                 0.2804         0.9951
max                1.0000                 1.0000         1.0000

Modified Logsdon breakdown (annotation-only + flank-corrected):
  Alu pass: 426 (was 0 original)
  L1/SVA pass: 183 (was 162 original)
  Not Alu/L1/SVA (auto-fail): 745
  Total pass: 609 (was 162 original)
  Difference from original: 447

  modified_logsdon_filter: 609 pass, 1001 fail (sampled 50/50)


## 8. Filter Evaluation: Copy Filtration (NO LONGER DONE)

Remove ALT elements whose locus already has the same element in the reference RepeatMasker run (F1_null). This identifies elements that are not novel insertions.

Remember that we ignored this!

## 9. Summary Table

I summarize the different filters that cause me to lose variants. Some of these are fairly reasonable (e.g. divergence), and the Logsdon criteria were always going to be fairly stringent, as there are only so many young TEs. From the summary below it's immediately clear that lots of records are lost to Bad_Tail_Position, and to some extent long Alus.

I ended up manually parsing through the entries that pass and fail different criteria to get a better sense.

In [11]:
all_summaries = flag_summaries + [tail_summary, logsdon_summary, mod_logsdon_summary]
summary_df = pd.DataFrame(all_summaries)
summary_df['pass_rate'] = (summary_df['total_pass'] / 
                           (summary_df['total_pass'] + summary_df['total_fail'])).round(3)

print(summary_df[['filter', 'total_pass', 'total_fail', 'pass_rate', 
                   'sampled_pass', 'sampled_fail']].to_string(index=False))

# Save summary
summary_df.to_csv(os.path.join(OUTPUT_DIR, "_filter_summary.tsv"), sep="\t", index=False)

                        filter  total_pass  total_fail  pass_rate  sampled_pass  sampled_fail
              flag_ALU_>500Len        1140         470      0.708            50            50
                 flag_ALU_>Len        1456         154      0.904            50            50
           flag_ALU_DIVERGENCE        1420         190      0.882            50            50
flag_Alu_Linker_Region_Warning        1579          31      0.981            50            31
        flag_Bad_Tail_Position         627         983      0.389            50            50
          flag_LINE_DIVERGENCE        1559          51      0.968            50            50
           flag_MULTI-ALU_Hits        1565          45      0.972            50            45
         flag_No_Tail_Detected        1515          95      0.941            50            50
     flag_OLDER_LINE_SUBFAMILY        1499         111      0.931            50            50
          tail_type_NoTailType        1356         254      

## 10. List Output Files

In [17]:
print("Output files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(fpath)
    print(f"  {f} ({size:,} bytes)")

Output files:
  .DS_Store (6,148 bytes)
  _filter_summary.tsv (1,938 bytes)
  copy_filtration_FAIL.tsv (38,847 bytes)
  copy_filtration_PASS.tsv (7,617 bytes)
  flag_ALU_DIVERGENCE_FAIL.tsv (36,129 bytes)
  flag_ALU_DIVERGENCE_FAIL_FILTERED.tsv (25,006 bytes)
  flag_ALU_DIVERGENCE_PASS.tsv (24,930 bytes)
  flag_ALU_DIVERGENCE_PASS_FILTERED.tsv (18,490 bytes)
  flag_ALU_gt500Len_FAIL.tsv (36,748 bytes)
  flag_ALU_gt500Len_FAIL_FILTERED.tsv (19,922 bytes)
  flag_ALU_gt500Len_PASS.tsv (34,279 bytes)
  flag_ALU_gt500Len_PASS_FILTERED.tsv (14,217 bytes)
  flag_ALU_gtLen_FAIL.tsv (24,248 bytes)
  flag_ALU_gtLen_FAIL_FILTERED.tsv (15,391 bytes)
  flag_ALU_gtLen_PASS.tsv (32,242 bytes)
  flag_ALU_gtLen_PASS_FILTERED.tsv (15,786 bytes)
  flag_Alu_Linker_Region_Warning_FAIL.tsv (4,319 bytes)
  flag_Alu_Linker_Region_Warning_FAIL_FILTERED.tsv (9,976 bytes)
  flag_Alu_Linker_Region_Warning_PASS.tsv (31,433 bytes)
  flag_Alu_Linker_Region_Warning_PASS_FILTERED.tsv (16,653 bytes)
  flag_Bad_Tail_Pos

In [13]:
pd.set_option('display.max_colwidth', None)#30)
F1_alt["Element Hits"]

0                                                                                                                                                                                        NONE
1                                                                                                                                                                                        NONE
2              ['236 17.1 23.2 1.7 chr1-28078-ALT 975 1069 (1374) C L1MD3 LINE/L1 (1336) 6659 6545 73 *', '18 3.9 7.4 0.0 chr1-28078-ALT 1235 1261 (1182) + (TA)n Simple_repeat 1 29 (0) 75']
3                                                                                                                                                                                        NONE
4                                                                                                                                                                                        NONE
                                                  

In [14]:
pd.set_option('display.max_colwidth', 300)
F1_alt[F1_alt["ID"].str.contains("9048657")]

,ID,Sequence,Element Hits,Sequence Length,Element Annotation,Element Divergence,Element Proportion,Element Percentage,Element Designation,Orientation,Flags,Tail Beginning,Tail Type,Tail Length,Tail Seed Hits,Unique Element Count,Twin Priming Flag,parsed_flags
17,chr1-9048657-ALT,CGGCGACAAAGTGAGACTCTGTCTCAAAAACAAAAACAAAAACAAATTCGGCGCATCCACCGCTGGCCAGCCTTTTCCTCTCAGAAGTCCCCTCTCTCTCACTAGAGAGAGAGCTGTGTTCCTCTTTCTTTCTCTTTCTCTTGCTTGTTGAACCTCCACTCCTAAACGCCTCATGTGTGTCCGTGTCCTAAGTTTTTCTGGGGCAAGACGATGAACCCCAGGGTATTTACCCCAGACAACGTAGCCGCTTCAGAAGGGTGGGCCTGGGACCCAGCACAGGGGGCAGAAGAGAAGGA...,"['845 11.6 10.1 0.0 chr1-9048657-ALT 425 553 (1767) + LTR8 LTR/ERV1 1 142 (549) 127', '1297 17.3 5.4 7.8 chr1-9048657-ALT 885 1006 (1314) C AluSx SINE/Alu (2) 310 209 129', '2803 1.9 0.0 0.0 chr1-9048657-ALT 1012 1318 (1002) + AluY SINE/Alu 1 307 (4) 130', '1297 17.3 5.4 7.8 chr1-9048657-ALT 190...",2320,LTR8,12.801709,"{'LTR8': 0.15129310344827587, 'AluSx': 0.13448275862068965, 'AluY': 0.13232758620689655}",0.418103,LTR/ERV1,+,['Bad_Tail_Position'],494,Possible_A-Tail*_and_Possible_T-Tail,6,31,Multiple_Element_Types_MultiDesignations,NONE,{Bad_Tail_Position}


In [15]:
F1_alt[F1_alt['Unique Element Count'].str.contains("One_Element_Type")]

,ID,Sequence,Element Hits,Sequence Length,Element Annotation,Element Divergence,Element Proportion,Element Percentage,Element Designation,Orientation,Flags,Tail Beginning,Tail Type,Tail Length,Tail Seed Hits,Unique Element Count,Twin Priming Flag,parsed_flags
5,chr1-54656-ALT,AGGAAAAGAGTACTGCTACTTTCATTTTCTTTTCTAACCACTGCCCTAGAAGAATAACACCTCACATTCTCTGGTAGACTGAAGCCTGCAGCTGCAAGGATGCTCAGAAAACCAGACTTCTATTCATTGGCCACTTGCTGCACGCTCTGTCCCTCGATCCTTGCAACGTACCTTTGAGGTAGGCACAAAAATCCCCATTTACAAGTGAGAAAACCAAGGCAAAAGAGGTTCAGTTTATTTCAGGCTCAAATGTGCTTCCAATAGCTTATAGCCTCTACAGCTAGTGTTTGAACCCA...,"['465 13.9 0.0 0.0 chr1-54656-ALT 1448 1519 (6357) C tRNA-Gln-CAA_v tRNA (0) 72 1 85', '432 13.4 1.5 0.0 chr1-54656-ALT 4567 4633 (3243) C tRNA-Ser-TCA(m) tRNA (5) 68 1 86']",7876,tRNA-Gln-CAA_v,13.900000,"{'tRNA-Gln-CAA_v': 0.009141696292534281, 'tRNA-Ser-TCA(m)': 0.0085068562722194}",0.017649,tRNA,-,['Bad_Tail_Position'],143,Possible_A-Tail*_and_Possible_T-Tail,6,30,One_Element_Type_MultiDesignations,NONE,{Bad_Tail_Position}
20,chr1-12360369-ALT,CTGGTGCCAGTTTTCAAAGGGAATGCTTCCCATTTTTGTCCATTCAGTATGATATTGGCTGTGGGTTTGTCATAGATGGCTCTTATTATCTTGAGATATGTCCCATCAATACCTAATTTACTGAGAGTTTTTAGCATGAAGGTTTTTGAATTTTGTCGAATGCCTTTTCTGCATCTATTGAGATAATCATATGGTTTTTGTCGTTGGTTCTGTTTACATGCTGGATTCTGTTTATTGATTTTCATATGCTGAACCAGCCTTGCATCCCAGGGATGAAGCCCACTTGATCATGGTGG...,"['9997 5.0 0.1 0.4 chr1-12360369-ALT 1 1004 (1272) C L1P1 LINE/L1 (2064) 4082 3082 42', '2073 1.7 0.0 0.4 chr1-12360369-ALT 1012 1275 (1001) C L1HS LINE/L1 (0) 6155 5915 43', '9997 5.0 0.1 0.4 chr1-12360369-ALT 1276 1528 (748) C L1P1 LINE/L1 (3066) 3080 2828 42', '3363 4.9 0.0 0.2 chr1-12360369-...",2276,L1P1,4.974599,"{'L1P1': 0.7403339191564148, 'L1HS': 0.11599297012302284}",0.856327,LINE/L1,-,No_Flags,32,Possible_A-Tail_and_Possible_T-Tail*,5,6,One_Element_Type_MultiDesignations,NONE,{}
21,chr1-15270195-ALT,AAAATACAAAAATTACTTGGGCGCAGTGGCACATGCTGGTAATCCCAGCTACTGAGGAGACTGAGGCAGGAGAATCGCTTGAACCTGGGAGGTGGAGGTGGCAGTGGGCCGAGATCACGCCACTGCACTCCAGCCTGGTGACAGAGCTAGACTGTGTCTCAAAAAAAAAAAAAAAAAATTAGCCGCCTGTGGTGGCGCATGCCTGTAATCCCAGCTACTTGGGAGGCTGAGGCAGAAGATTGCTTGAACCTGGGAGGCTGCAGCTATAATAAGCCGAGATTGCACCACTGCACTCC...,"['2353 9.1 0.7 0.3 chr1-15270195-ALT 945 1253 (4232) + AluSx SINE/Alu 1 310 (2) 49', '498 14.7 2.7 0.3 chr1-15270195-ALT 1254 1334 (4151) + FLAM_C SINE/Alu 2 80 (53) 48', '2130 9.5 2.7 0.0 chr1-15270195-ALT 1737 2030 (3455) C AluSx SINE/Alu (10) 302 1 51', '2182 11.8 0.7 0.3 chr1-15270195-ALT 20...",5485,AluSx,10.908810,"{'AluSx': 0.35387420237010025, 'FLAM_C': 0.014767547857793984}",0.368642,SINE/Alu,+,"['Bad_Tail_Position', 'ALU_>500Len', 'ALU_DIVERGENCE']",391,Possible_A-Tail_and_Possible_T-Tail*,18,82,One_Element_Type_MultiDesignations,NONE,"{ALU_>500Len, ALU_DIVERGENCE, Bad_Tail_Position}"
27,chr1-29716440-ALT,AATGGAAATTACAGAATTGAAAGATAATATAGTAACAAAGTAAAGATTTGTTGGATGGGCTTACCAGTAGAGTGAAGATGGCAAAACAGTGAAACAGTGAATTGGAGGACAGATCAATAGAATTTACCAATCTGAACAACAAAGAAAAAAGAGATGAATAAAATAAATAAACTCTCAGGGACCTGTGGGGTAATGGTTGAAAACATCCCAAATGATATAGATAGATTGACAGCAAGAGAATGGAAAAAGACATATAAGGCAAAAACTCATTGAAATAAAAGGCAGGGGTGGCTATA...,"['4052 19.9 7.5 1.1 chr1-29716440-ALT 1 1319 (1074) + L1MDb LINE/L1 1175 3709 (2697) 175', '239 17.4 0.0 4.3 chr1-29716440-ALT 1312 1383 (1010) + L1M4b LINE/L1 1855 1923 (4944) 176 *', '2425 18.4 2.9 2.9 chr1-29716440-ALT 1427 1985 (408) + L1MDb LINE/L1 3719 4277 (2129) 175', '533 19.3 0.7 11.0 ...",2393,L1MDb,19.442864,"{'L1MDb': 0.8432929377350606, 'L1M4b': 0.030087755954868366}",0.870038,LINE/L1,+,"['Bad_Tail_Position', 'OLDER_LINE_SUBFAMILY', 'LINE_DIVERGENCE']",89,Possible_A-Tail*_and_Possible_T-Tail,13,43,One_Element_Type_MultiDesignations,NONE,"{OLDER_LINE_SUBFAMILY, Bad_Tail_Position, LINE_DIVERGENCE}"
37,chr1-50096038-ALT,CCCATCTTAAAAAAAAAACTCGCTCTTTACCGTAACTCCTATTATTACCCCATTTCTCATTTTCCCTCAGCAAGAAAACTTCTTAAAGTATAGTGATATTCACTGCCCAATTTCTCTTTTCCCATTCTCCCTTGAACTTTAGTAAGATCTGCTTCCCCACTACTCTTCCCAAACTGCTCTTGTCAAAGTCACCAGTGATCTCCCCATTGCCAAATCCAAATGGCAAGTCTCAGTCCTCGTGGTACTCAATCTTGCAACTGTATTTGTCTCAGCCAATCACCTTCTCCTC

In [16]:
len(F1_alt)

1610

## Checking what matters

We shouldn't really care about a filter unless it passes the Logsdon criteria first. Recall that the filters were performed on the raw dataset, when in reality we want to compose these filters. Given that an element is Logsdon-valid, how many still have problems from other flags?

In [17]:
passLogsdon=F1_alt[mod_logsdon_pass]
passLogsdon[passLogsdon['Unique Element Count'].str.contains("One_Element_Type_MultiDes")]
#passLogsdon[passLogsdon['Element Designation'].str.contains("LINE")]

,ID,Sequence,Element Hits,Sequence Length,Element Annotation,Element Divergence,Element Proportion,Element Percentage,Element Designation,Orientation,Flags,Tail Beginning,Tail Type,Tail Length,Tail Seed Hits,Unique Element Count,Twin Priming Flag,parsed_flags
20,chr1-12360369-ALT,CTGGTGCCAGTTTTCAAAGGGAATGCTTCCCATTTTTGTCCATTCAGTATGATATTGGCTGTGGGTTTGTCATAGATGGCTCTTATTATCTTGAGATATGTCCCATCAATACCTAATTTACTGAGAGTTTTTAGCATGAAGGTTTTTGAATTTTGTCGAATGCCTTTTCTGCATCTATTGAGATAATCATATGGTTTTTGTCGTTGGTTCTGTTTACATGCTGGATTCTGTTTATTGATTTTCATATGCTGAACCAGCCTTGCATCCCAGGGATGAAGCCCACTTGATCATGGTGG...,"['9997 5.0 0.1 0.4 chr1-12360369-ALT 1 1004 (1272) C L1P1 LINE/L1 (2064) 4082 3082 42', '2073 1.7 0.0 0.4 chr1-12360369-ALT 1012 1275 (1001) C L1HS LINE/L1 (0) 6155 5915 43', '9997 5.0 0.1 0.4 chr1-12360369-ALT 1276 1528 (748) C L1P1 LINE/L1 (3066) 3080 2828 42', '3363 4.9 0.0 0.2 chr1-12360369-...",2276,L1P1,4.974599,"{'L1P1': 0.7403339191564148, 'L1HS': 0.11599297012302284}",0.856327,LINE/L1,-,No_Flags,32,Possible_A-Tail_and_Possible_T-Tail*,5,6,One_Element_Type_MultiDesignations,NONE,{}
37,chr1-50096038-ALT,CCCATCTTAAAAAAAAAACTCGCTCTTTACCGTAACTCCTATTATTACCCCATTTCTCATTTTCCCTCAGCAAGAAAACTTCTTAAAGTATAGTGATATTCACTGCCCAATTTCTCTTTTCCCATTCTCCCTTGAACTTTAGTAAGATCTGCTTCCCCACTACTCTTCCCAAACTGCTCTTGTCAAAGTCACCAGTGATCTCCCCATTGCCAAATCCAAATGGCAAGTCTCAGTCCTCGTGGTACTCAATCTTGCAACTGTATTTGTCTCAGCCAATCACCTTCTCCTCCTTGAAA...,"['28681 1.4 0.0 0.0 chr1-50096038-ALT 1003 7033 (1348) + L1HS LINE/L1 1 6032 (0) 236', '735 14.4 0.7 1.4 chr1-50096038-ALT 7169 7309 (1072) C L1M3 LINE/L1 (798) 5342 5203 238', '52 3.4 0.0 0.0 chr1-50096038-ALT 7312 7372 (1009) + (A)n Simple_repeat 1 61 (0) 239']",8381,L1HS,1.400000,"{'L1HS': 0.7196038658871257, 'L1M3': 0.01682376804677246, '(A)n': 0.007278367736546951}",0.743706,LINE/L1,+,No_Flags,30,Possible_A-Tail*_and_Possible_T-Tail,5,138,One_Element_Type_MultiDesignations,FLAG,{}
55,chr1-82984359-ALT,GGGCAGCCGTGGTTTATTGGAGGTGTGGTTAAAGCACTCATGGTCTTTGTTCCTTTCCCAGTCTGAGGGCAGCAAAGGAAGTACCACTGTGGTTACAGTGGCAGAGGAGTTTTCAGTTGCCTCTTGGAGCTTCACCTCAAAGAAATACAAGGATCCTGCTAACCGGAATGTTCAGCTGGGTGTGGCTGAATATTCATCCAGGGACCCTGCTTGGTGAAAAGCTGGGGGTCAAGGCTCACAGGAAGGGGTGACTATACTCCTATCTGTATGGTGACTGTGGCAAGCTGGAAATGTGA...,"['8610 1.1 0.0 0.0 chr1-82984359-ALT 1005 2742 (1003) C L1HS LINE/L1 (4) 6151 4414 349', '652 19.6 0.8 0.2 chr1-82984359-ALT 2746 2901 (844) C L1P4b LINE/L1 (6477) 1684 1493 350', '1648 18.2 1.0 1.3 chr1-82984359-ALT 2910 3215 (530) C L1P4b LINE/L1 (7021) 1140 836 350']",3745,L1HS,1.100000,"{'L1HS': 0.46408544726301737, 'L1P4b': 0.1233644859813084}",0.587450,LINE/L1,-,['Bad_Tail_Position'],145,Possible_A-Tail*_and_Possible_T-Tail,12,6,One_Element_Type_MultiDesignations,NONE,{Bad_Tail_Position}
88,chr1-190350151-ALT,GACCCAAATTTATTATCTCAGAGTTTCTTTGGGTCAGCAGTTCAACACATGTGAATGAGTTCTCAGTTCCGGTTCTCATAAGATCTGTATCAAGGTGTTGACTGCTGTCTTCACTGAGGCTCAATGAGGTTCATTGAGGTTCATTGAGTCTGTTCCAAGGTCTCTGAAATTGTGGACACTTAATTTTTTTTCTTGTCTTGTGGCTAGAGATCAATCTTAGTACCTAGGGGCCACTCTCAAGTAAGTTAACTGATTTGATACTTTAATTAAATATGAGAAATTCCTATTTATAAAAA...,"['7892 0.5 0.0 0.6 chr1-190350151-ALT 685 1901 (1174) C L1HS LINE/L1 (0) 6155 5283 512', '1665 15.1 2.3 0.6 chr1-190350151-ALT 703 1006 (2069) C L1MA2 LINE/L1 (0) 6304 5996 511', '1384 3.0 0.0 0.0 chr1-190350151-ALT 1901 2066 (1009) + L1HS LINE/L1 5116 5281 (865) 512 *', '4954 10.5 2.0 2.2 chr1-...",3075,L1HS,0.800072,"{'L1HS': 0.44943089430894306, 'L1MA2': 0.09886178861788618, 'L1MA1': 0.3300813008130081}",0.777561,LINE/L1,-,['Bad_Tail_Position'],183,Possible_A-Tail_and_Possible_T-Tail*,8,6,One_Element_Type_MultiDesignations,FLAG,{Bad_Tail_Position}
280,chr12-43433911-ALT,CCACCCCAAATCAACAGAATATACATTTTTTTCAGCACCACACCACACCTATTCCAAAATTGACCACATACTTGGAAGTAAAGCTCTCCTCAGCAAATGTAAAAGAACAGAAATTATAACAAACTATCTCTCAGACCACAGTGCAATCAAACTAGAACTCAGGATTAAGAATCTCACTCAAAGCCGCTCAACTACATGGAAACTGAACAACCTGCTCCTGAATGACTACTGGGTACATAACGAAATGAAGGCAGAAATAAAGATGTTCTTTGAAACCAACGAGAACAAAGACACAA...,"['11962 2.0 0.0 0.1 chr12-43433911-ALT 1 1403 (1044) + L1P1 LINE/L1 2661 4061 (2085) 

In [18]:
passLogsdon[passLogsdon['Flags'].str.contains("MULTI-ALU_Hits")]

,ID,Sequence,Element Hits,Sequence Length,Element Annotation,Element Divergence,Element Proportion,Element Percentage,Element Designation,Orientation,Flags,Tail Beginning,Tail Type,Tail Length,Tail Seed Hits,Unique Element Count,Twin Priming Flag,parsed_flags
285,chr12-55412921-ALT,TTCCCAGGCCCAGAAGACTGATGATATTTGCAGTAATTTTTTATACGCAAAGCAGGTGAAAAAGCATTCATATAACAAACTAATTGGTATCTTATAAGGAGATACAATTTCATTGAGTGATGGAGAAATAGCAAGAACCCTAGAGACTGTGTTGTCAGATAAGACATATGTTTGTAGCTACCATTATAAATAATTAATGCAGGCTAAGGACTGGAAGCAGTTATTGCTCCATGAACAGAAATAAACTCATTTCCTCTTGCTGAAGAATGTCATAATGGTCAGAGATGACATGAACA...,"['1584 12.4 0.4 0.4 chr12-55412921-ALT 774 1000 (1247) C AluSx SINE/Alu (8) 304 78 1453', '2785 1.3 0.0 0.3 chr12-55412921-ALT 1001 1304 (943) C AluY SINE/Alu (8) 303 1 1454']",2247,AluY,1.3,"{'AluSx': 0.10102358700489542, 'AluY': 0.1352914997774811}",0.236315,SINE/Alu,-,"['MULTI-ALU_Hits', 'ALU_>500Len']",36,Possible_A-Tail_and_Possible_T-Tail*,6,10,One_Element_Type_MultiDesignations,NONE,"{ALU_>500Len, MULTI-ALU_Hits}"
916,chr3-69330339-ALT,CATCCAACTTTGAACAGATACCTTTGTATTAGAAACTGGGTTCACCTTGTTTTTGGAGTCCAGGCCCCTGACTACATCCTAGTCCCAGTAAACAGGTCCCTATCTTATTTCCCACTACTTTCTGGTGACCAGCCCTGGTGCCTCCTGCCTAGGCATCAGATTACCAGCTTTCTGAATCCACAATTCCATCTCTCCCTCAAGGGGGCCATTGTAATGGCTTCATATTTATGGAGTGTATGAAAAGTACATATGGGCTTTGGGGAAAATGGCTCTTCAGGCACACCTATCAACACAAA...,"['2965 0.6 0.0 0.0 chr3-69330339-ALT 1011 1323 (1003) + AluYb8 SINE/Alu 1 313 (5) 4716', '2169 12.5 0.0 0.0 chr3-69330339-ALT 1501 1796 (530) + AluSz6 SINE/Alu 12 307 (5) 4717']",2326,AluYb8,0.6,"{'AluYb8': 0.1345657781599312, 'AluSz6': 0.12725709372312985}",0.261823,SINE/Alu,+,"['MULTI-ALU_Hits', 'ALU_>500Len']",49,Possible_A-Tail_and_Possible_T-Tail*,5,46,One_Element_Type_MultiDesignations,NONE,"{ALU_>500Len, MULTI-ALU_Hits}"
1329,chr7-52327606-ALT,AGTGCAGTGGCGCAATCTCAGCTCACTGTAACCTCCGCCTCCTGAGTTCAAGTGATTCTCTTGCCTCAGCCTTTTGAGTAGCTGGGATTACAGGCACACGCCACCATGTCCAGCTAATTTTTGTATTTTTAGTAGAGACGGGATTTCACCATATTGGTCAGGCTAGTCTTGGACTCCTGACCTCGTGATCCGCCCGAGTCGGCCTCCCAAAGTGCTGGAATTACAGGCGTGAGCCACCGTGCCTGGCCCTGTTTTTCCTTTTTATCCTCAATCTTGTAATGTGTCTTGTTTTGAGA...,"['2719 2.6 0.0 2.2 chr7-52327606-ALT 1002 1319 (1010) + AluY SINE/Alu 1 311 (0) 6593', '2051 11.9 0.0 1.8 chr7-52327606-ALT 1322 1612 (717) C AluSx SINE/Alu (26) 286 1 6594']",2329,AluY,2.6,"{'AluY': 0.1365392872477458, 'AluSx': 0.12494632889652212}",0.261486,SINE/Alu,+,"['Bad_Tail_Position', 'MULTI-ALU_Hits', 'ALU_>500Len']",1008,Possible_A-Tail*_and_Possible_T-Tail,32,37,One_Element_Type_MultiDesignations,NONE,"{ALU_>500Len, Bad_Tail_Position, MULTI-ALU_Hits}"


In [20]:
no_tails=passLogsdon[passLogsdon['Flags'].str.contains("No_Tail")]
no_tails

,ID,Sequence,Element Hits,Sequence Length,Element Annotation,Element Divergence,Element Proportion,Element Percentage,Element Designation,Orientation,Flags,Tail Beginning,Tail Type,Tail Length,Tail Seed Hits,Unique Element Count,Twin Priming Flag,parsed_flags
486,chr16-62535341-ALT,GGCGGGGGGCTGACCCCCCAACCTCCCTCCCGGACGGGGCGGCTGGCCGGGCAGAGGGGCTCCTCACTTCCCAGTAGGGGCGGCCGGGCAGAGGCGCCCCTCACCTCCCGGACGGGGCGGCTGGCCGGGCGGGGGGGCTGACTCCCCCCACCTCCCTCCCGGACGGGGCGGCTGGCCGGGCGGGGGGCTGACACCCCCACCTCCCTCCCGGACGGGGCGGCTGGCCGGGCAGAGGGGCTCCTCACTTCCCAGTAGGGGCGGCCGGGCAGAGGCGCCCCTCACCTCCCGGACGGGGC...,['2524 7.3 1.8 9.6 chr16-62535341-ALT 1 2359 (0) C SVA_A Retroposon/SVA (515) 872 389 2584'],2359,SVA_A,7.3,{'SVA_A': 1.0},1.0,Retroposon/SVA,-,['No_Tail_Detected'],No_Tail_Detected,No_Tail_Type,0,0,One_Element,NONE,{No_Tail_Detected}


In [22]:
bad_tails=passLogsdon[passLogsdon['Flags'].str.contains("Bad")]
len(bad_tails)

445

In [23]:
AluLen=passLogsdon[passLogsdon['Flags'].str.contains("Len")]
len(AluLen)

425

## Conclusions

The real issues appear to be the presence of bad tail and Alu length flags. What could have caused this?

Alus exist in a relatively narrow band of lengths. If you look at the length of the actually identified Alu sequences that were flagged, they actually seem to be totally fine. The only thing that came to mind was that the *sequence length*, not the *identified TE length*, was causing the problem.

That could also explain the bad_tails, if L1ME-AID assumes that the tail must be at the end of the *sequence*, not the *identified TE positions*.

This analysis ended up informing why we ignore the flanking sequence, which caused Juan Macias and I way more headaches than we needed. Much of what we learned here is that it is largely worth trusting PanGenie! If it thinks that's where the ALT INS is, that's probably where the ALT INS is!